# Демо на защите — ozbekova, Citi Bike DWH

Запускать сверху вниз (Shift+Enter). Пароли нигде не печатаются: берутся из переменных окружения стенда.

In [ ]:
import os, re, subprocess, time, warnings
import pandas as pd, psycopg2, requests

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# переменные PG*/CH_* стенда живут в login-shell; переносим их в окружение ноутбука, не печатая
for line in subprocess.run(["bash", "-lic", "env"], capture_output=True, text=True).stdout.splitlines():
    m = re.match(r"((PG|CH_)[A-Z_]*)=(.*)", line)
    if m:
        os.environ[m.group(1)] = m.group(3)

conn = psycopg2.connect(); conn.autocommit = True
def q(sql, params=None):
    return pd.read_sql(sql, conn, params=params)
print("подключено как", os.environ["PGUSER"])

## 1. CDC: меняю заявку в «базе сайта» — через ~10 секунд она меняется в ods

In [ ]:
TICKET = 2
q("""select 'сайт' as где, ticket_id, status, updated_at from ozbekova.maintenance_tickets where ticket_id = %(t)s
     union all
     select 'ods', ticket_id, status, updated_at from ozbekova_ods.ods_maintenance_tickets where ticket_id = %(t)s""",
  {"t": TICKET})

In [ ]:
# переключаем статус: in_progress <-> closed
with conn.cursor() as cur:
    cur.execute("""update ozbekova.maintenance_tickets
                   set status = case when status = 'closed' then 'in_progress' else 'closed' end, updated_at = now()
                   where ticket_id = %s returning status""", (TICKET,))
    print("на сайте теперь:", cur.fetchone()[0])
for sec in range(1, 31):
    time.sleep(1)
    ods = q("select status from ozbekova_ods.ods_maintenance_tickets where ticket_id = %(t)s", {"t": TICKET}).status[0]
    site = q("select status from ozbekova.maintenance_tickets where ticket_id = %(t)s", {"t": TICKET}).status[0]
    if ods == site:
        print(f"через {sec} с в ods: {ods}  ✓ Debezium → Kafka → JDBC sink")
        break
else:
    print("за 30 с не дошло — проверить статус коннекторов")

## 2. Идемпотентность: перезапуск за ту же дату не создаёт дублей

1. Выполнить ячейку ниже (запомнить числа).
2. В Airflow: `ozbekova.final_ozbekova_datamart` → запуск за 2026-10-20 → **Clear**.
3. Через минуту выполнить ячейку ещё раз — числа те же, `_updated_at` новее.

In [ ]:
DS = "2026-10-20"
q("""select 'ods_station_status' as таблица, count(*) as строк, null::timestamptz as обновлено
       from ozbekova_ods.ods_station_status where status_date = %(ds)s
     union all select 'dm_daily_station_activity', count(*), max(_updated_at)
       from ozbekova_datamart.dm_daily_station_activity where activity_date = %(ds)s
     union all select 'dm_station_daily', count(*), max(_updated_at)
       from ozbekova_datamart.dm_station_daily where activity_date = %(ds)s""", {"ds": DS})

## 3. Ответ на вопрос одним запросом к витрине: сколько поездок отнимает дождь

In [ ]:
q("""select day_type, rain_level, days_cnt, avg_trips,
            to_char(change_vs_dry * 100, 'FM990.0') || '%' as "к сухому дню",
            to_char(casual_change_vs_dry * 100, 'FM990.0') || '%' as "разовые",
            to_char(member_change_vs_dry * 100, 'FM990.0') || '%' as "подписчики"
     from ozbekova_datamart.dm_rain_impact_dbt
     order by day_type, case rain_level when 'dry' then 1 when 'rain' then 2 else 3 end""")

## 4. Журнал загрузок etl_load_log: каждая попытка каждой задачи

In [ ]:
q("""select dag_id, task_id, ds, try_number, status, rows_count,
            round(extract(epoch from finished_at - started_at)::numeric, 1) as сек
     from ozbekova_stg.etl_load_log order by id desc limit 12""")

## 5. Копия витрины в ClickHouse

In [ ]:
r = requests.post(f"http://{os.environ['CH_HOST']}:{os.environ['CH_PORT']}/",
                  params={"query": "select activity_date, snapshots_cnt, empty_snapshots, empty_share "
                                   "from ozbekova.dm_daily_station_activity final order by activity_date format TSVWithNames"},
                  headers={"X-ClickHouse-User": os.environ["CH_USER"], "X-ClickHouse-Key": os.environ["CH_PASSWORD"]},
                  timeout=10)
print(r.text)